> Find clusters of windows using graphs (*communities*)

In [ ]:
import pathlib
import random
import itertools

import sklearn.neighbors
import igraph as ig
import leidenalg

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from bokeh.palettes import Category20
from bokeh.plotting import show
from bokeh.io import output_notebook
from bokeh.resources import INLINE
from bokeh.models import Label

from IPython.display import display, Markdown

import trapallada.plot
import trapallada.analysis

In [ ]:
output_notebook(resources=INLINE)
# output_notebook()

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

# Input Files

Relative path and *Base* file name for the results (trained model). It should be a `str`.

In [ ]:
# input_files_stem = '../results/4_sources/vanilla_5000_epochs' # rotated "C" below
# input_files_stem = '../results/epochs/20000/4_sources/epochs' # <---------
# input_files_stem = '../results/4_sources/vanilla_5000_epochs_05_x_sd'
# input_files_stem = '../results/4_sources/fft'
# input_files_stem = '../results/4_sources/fft_x_sd_2'
# input_files_stem = '../results/4_sources/fft_x_sd_10'
# ===============
# input_files_stem = '../results/new/4_sources/vanilla'
# input_files_stem = '../results/new/4_sources/fft_vanilla'
input_files_stem = '../results/new/4_sources/fft_vanilla_x_sd_2'

It is turned into a `Pathlib`

In [ ]:
input_files_stem = pathlib.Path(input_files_stem)
input_files_stem

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [ ]:
ds, positions_df, model, results_df, parameters, analysis_df = trapallada.analysis.load_results_dir(input_files_stem, analysis_also=True)

Files to *cache* the construction of the graph

In [ ]:
nodes_file = input_files_stem / 'nodes.parquet'
edges_file = input_files_stem / 'edges.parquet'

# Processing

For the sake of convenience

In [ ]:
zs_columns = [f'z_{i}' for i in range(1, parameters['model']['dim_z']+1)]
zs_columns[:7]

The *adjacency* matrix (distances to non-neighbors are zero)

In [ ]:
n_neighbors = 10
# n_neighbors = 5
# adj_matrix = sklearn.neighbors.kneighbors_graph(analysis_df[zs_columns].values, n_neighbors=n_neighbors, mode='distance')
adj_matrix = sklearn.neighbors.kneighbors_graph(analysis_df[zs_columns].values, n_neighbors=n_neighbors, mode='distance', metric='cosine')
adj_matrix.shape

**Similarity** metric (as suggested by JCS)

In [ ]:
# alpha = 1.
# adj_matrix.data = np.exp(- alpha * adj_matrix.data)

In [ ]:
adj_matrix.data =  1. - adj_matrix.data

A function to build an [igraph](https://igraph.org/) graph from a given similarity matrix

In [ ]:
def similarity_matix_to_graph(M: csr_matrix) -> ig.Graph:
    matrix_coo = M.tocoo()
    edges = list(zip(matrix_coo.row, matrix_coo.col))
    weights = matrix_coo.data

    g = ig.Graph(edges=edges, directed=False)
    g.es['weight'] = weights

    return g

G = similarity_matix_to_graph(adj_matrix)

Leidgen algorithm using [leidenalg](https://leidenalg.readthedocs.io/en/stable/index.html) implementation.

In [ ]:
partition = leidenalg.find_partition(G, leidenalg.ModularityVertexPartition)
partition

Commnities are *extracted* from the resulting partition, and its number computed

In [ ]:
communities = partition.membership
n_communities = max(communities) + 1
n_communities

If `DataFrame`s for nodes and edges are found, they are loaded (otherwise, the corresponding variables are set to `None`)

In [ ]:
nodes_df = pd.read_parquet(nodes_file) if nodes_file.exists() else None
if nodes_df is not None:
    print(f'found "{nodes_file}"')

edges_df = pd.read_parquet(edges_file) if edges_file.exists() else None
if edges_df is not None:
    print(f'found "{edges_file}"')

A `DataFrame` with information on the nodes. Different *layout*s are implemented in [igraph](https://python.igraph.org/en/stable/visualisation.html#graph-layouts) (see also [the corresponding class](https://python.igraph.org/en/stable/api/igraph.Graph.html#layout) for details).

In [ ]:
if nodes_df is None:

    # so that, for the same data, the results are consistent
    random.seed(42)

    # the *layout* of the nodes when plotting the graph
    layout = G.layout('auto')
    # layout = G.layout('kamada_kawai')

    # colors palette
    palette = Category20[20]
    
    nodes_df = pd.DataFrame(
        {
            'community': communities,
            'fill_color': [palette[c % 20]for c in communities],
        },
        index=analysis_df.index
    )

    nodes_df[['x', 'y']] = np.array(layout.coords)
    nodes_df['size'] = 8
    nodes_df['alpha'] = 1.

    unknown_series = analysis_df['main_type'].isna()
    nodes_df.loc[unknown_series, 'marker'] = 'square'
    nodes_df.loc[~unknown_series, 'marker'] = 'diamond'

nodes_df

Another one for the edges

In [ ]:
if edges_df is None:

    edges_df = pd.DataFrame(
        [(edge.source, edge.target) for edge in G.es], columns=['start', 'end']
    )

edges_df

The above `DataFrame`s are saved to disk

In [ ]:
nodes_df.to_parquet(nodes_file)
edges_df.to_parquet(edges_file)

# Overall view

In [ ]:
# fig = trapallada.analysis.plot_graph(analysis_df, nodes_df, edges_df)
fig = trapallada.analysis.plot_graph(nodes_df, edges_df, analysis_df)
show(fig)

In [ ]:
n_samples = 10

# A few eyeball-selected regions

A *wrapper* function around `trapallada.analysis.inspect_regions`, which does the heavy lifting

In [ ]:
inspect_regions = lambda fig, n_samples, x_range_list, y_range_list, labels:\
      trapallada.analysis.inspect_regions(
          fig, n_samples, x_range_list, y_range_list, labels,
          model, ds, nodes_df, analysis_df)

Zooming in on a few clusters

In [ ]:
# regions = [
#     {
#         'x_range': (222, 228),
#         'y_range': (-170, -162),
#     },
#     {
#         'x_range': (-159, -152),
#         'y_range': (44, 58),
#     },
#     {
#         'x_range': (30, 40),
#         'y_range': (-210, -200),
#     },
# ]
regions = [
    {
        'x_range': (230, 255),
        'y_range': (220, 260),
    },
    {
        'x_range': (-40, 100),
        'y_range': (370, 400),
    },
    {
        'x_range': (-320, -270),
        'y_range': (0, 70),
    },
    {
        'x_range': (-80, 0),
        'y_range': (-10, 50),
    },
]

x_range_list = list(map(lambda x: x['x_range'], regions))
y_range_list = list(map(lambda x: x['y_range'], regions))
labels = [f'Region {i}' for i in range(len(x_range_list))]

inspect_regions(
    trapallada.analysis.plot_graph(nodes_df, edges_df, analysis_df),
    n_samples,x_range_list, y_range_list, labels)


In [ ]:
# trapallada.analysis.predict_and_plot(model, ds, 47365, display_metadata=False)

In [ ]:
# sample_and_plot_range(n_samples, x_range, y_range)

# Cataclysmic

A convenience to plot all the windows for a given signal (in the above space)

In [ ]:
def plot_signal(name: str):
    
    select = (analysis_df['signal'] == name)

    fig = trapallada.analysis.plot_graph(nodes_df, edges_df, analysis_df)

    # fig.renderers[0].node_renderer.level
    
    # for the sake of readability
    data = fig.renderers[0].node_renderer.data_source.data
    
    for i in analysis_df[select].index.tolist():
        data['marker'][i] = 'circle'
        # data['size'][i] = 10
        data['fill_color'][i] = 'magenta'

    for i in analysis_df[~select].index.tolist():
        data['alpha'][i] = 0.0
    
    return fig, analysis_df[select]

In [ ]:
fig, select_df = plot_signal('0840380201/P0840380201PNS003SRCTSR8004')
show(fig)
# display(select_df)

# Javi's paper

All the associated windows

In [ ]:
fig, select_df = plot_signal('0605950101/P0605950101PNS003SRCTSR8001')
show(fig)

Zooming in on a few clusters

In [ ]:
select = (analysis_df['signal'] == '0605950101/P0605950101PNS003SRCTSR8001')
fig = trapallada.analysis.plot_graph(nodes_df[select], edges_df, analysis_df[select])

# x_range_list = [
#     (-22, 5),
#     (115, 140),
#     (-195,-180),
#     (-100,60),
# ]
# y_range_list = [
#     (-10, 5),
#     (-170, -155),
#     (-180, -170),
#     (-240, -190)
# ]

# x_range_list = [
#     (250, 290),
#     (210, 230),
#     (105, 120),
# ]
# y_range_list = [
#     (140, 210),
#     (40, 70),
#     (30, 45),
# ]

x_range_list = [
    (250, 290),
    (210, 230),
]
y_range_list = [
    (140, 210),
    (40, 70),
]

labels = [f'Region {i}' for i in range(len(x_range_list))]

inspect_regions(fig, n_samples,x_range_list, y_range_list, labels)